# RAG in a few lines

An AI model has never read *your* documents, and when it doesn't know something it tends to make
it up. **Retrieval-augmented generation (RAG)** fixes that: before the model answers, we look up
the relevant passages and hand them over together with the question.

Run the cells from top to bottom with **Shift + Enter**. You need an [Aitta](https://aitta.csc.fi/public)
API key in a `.env` file (see the README).

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()  # reads AITTA_API_KEY from the .env file

# Any OpenAI-compatible service works: change the address, the key and the model names.
aitta = dict(base_url="https://aitta-api.csc.fi/openai/v1", api_key=os.environ["AITTA_API_KEY"])
llm = ChatOpenAI(model="google/gemma-4-26B-A4B-it", timeout=300, **aitta)
embeddings = OpenAIEmbeddings(model="Qwen/Qwen3-Embedding-8B", check_embedding_ctx_length=False, **aitta)


def load_page(path):
    """Read one saved web page. Its third line holds the page's address."""
    text = path.read_text(encoding="utf-8")
    return Document(text, metadata={"url": text.splitlines()[2].removeprefix("Source: ")})


def show(text):
    display(Markdown(text))


print("Waking up the models. If nobody has used them lately, this can take a few minutes...")
llm.invoke("Hi")
embeddings.embed_query("Hi")
print("Ready.")

## 1. Ask the model on its own

In [ ]:
question = "Which GPUs will the new LUMI-AI supercomputer use, and who is building it?"

show(llm.invoke(question).content)

It sounds convincing. But when we tried it, the model named NVIDIA GPUs — while the LUMI AI
Factory website says LUMI-AI is being built by **Bull** with **AMD Instinct MI430X** GPUs. The
model has never read that website, so it fills the gap with something plausible.

## 2. Give the model the website

The folder `lumi_aif_website/` holds a copy of the LUMI AI Factory website. We cut every page
into short passages and **index** them by meaning: an *embedding model* turns each passage into
a list of numbers, so that passages about similar things get similar numbers — even when they
use different words.

In [ ]:
pages = [load_page(path) for path in Path("lumi_aif_website").glob("*.md")]
passages = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(pages)
library = InMemoryVectorStore.from_documents(passages, embeddings)

print(f"{len(pages)} pages cut into {len(passages)} passages and indexed.")

Now the RAG part: **search** for the passages that best match the question, **put them in front
of the question**, and let the model **answer**. Same question as before:

In [ ]:
def search(query: str) -> str:
    """Search the LUMI AI Factory website. Returns the best-matching passages with their links."""
    print("🔎", query)
    found = library.similarity_search(query, k=4)
    return "\n\n".join(f"Source: {p.metadata['url']}\n{p.page_content}" for p in found)


def ask_with_docs(question):
    passages = search(question)
    prompt = f"Answer using only these passages, and end with the source links.\n\n{passages}\n\nQuestion: {question}"
    return llm.invoke(prompt).content


show(ask_with_docs(question))

**That is RAG.** Compare it with the first answer: now the model answers from the website and
tells you where to read more. Try your own question:

In [ ]:
show(ask_with_docs("How many GPU hours does the Playground access give?"))

## 3. Let the model search for itself

Above, we searched **once**, with the question's exact words. Questions that need facts from
many pages need more than one search. So instead, we hand the model the `search` function and
let it decide what to look up, and how many times. A model that uses tools like this is called an
**agent**. Each 🔎 line below is a search the model chose to run.

In [ ]:
def ask_agent(question):
    model = llm.bind_tools([search])  # tell the model it may use the search function
    messages = [HumanMessage(question + "\n\nSearch as often as you need. Answer only from what "
                             "you found, and end with the source links.")]
    for _ in range(8):                 # at most 8 rounds
        reply = model.invoke(messages)
        messages.append(reply)
        if not reply.tool_calls:       # no more searches requested: this is the answer
            return reply.content
        for call in reply.tool_calls:  # run each search the model asked for
            messages.append(ToolMessage(search(**call["args"]), tool_call_id=call["id"]))
    return "Stopped after 8 rounds without an answer."


show(ask_agent("List every way to get computing access through LUMI AI Factory, "
               "with how many GPU hours each gives and for how long."))

## What next?

- **Your own documents:** put your own markdown files in a folder and point the "Give the model the website" cell at it.
- **The advanced version:** `02_agentic_rag.ipynb` turns the agent into a full assistant that remembers the conversation, asks back when a question is unclear, and researches several sub-questions in parallel.